### Experiment with gpt o1

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../libs'))

from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import prompts as prompts
from exp_utils import classify_dataset, evaluate_experiment
from openai_lib import call_openai
from tqdm import tqdm
import json
import re


In [ ]:
# Define model name
MODEL_NAME = "o1"

### Dataset load

In [ ]:
# Load ground truth bug locations
with open("../../../Datasets/bugLocationDappScan.json", "r") as f:
    bug_locations = json.load(f)

In [ ]:
def normalize_path(path):
    if isinstance(path, str):
        return path.replace('\\', '/').replace('./', '').replace('.\\', '')
    return path

bugloc_dict = {
    normalize_path(item['file']): normalize_path(item['location'])
    for item in bug_locations
}


# Load the dataset and convert to appropriate data types
csv_path = "../../../Datasets/error_detection_task_dataset.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset = df.reset_index(drop=True)
#subset.head()

#### Zero-shot Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

##### Classificação de erros pela LLM usando taxonomia

Nesta seção, vamos passar a taxonomia de erros para a LLM e pedir para ela classificar os casos de erro, comparando com as labels do dataset.

In [ ]:
# Extração da taxonomia de categorias SWC (baseado em distribuicao.py)
category_list = []
real_base = '../../../Datasets/DAppSCAN-main/DAppSCAN-source/'
for item in bug_locations:
    relative_location = item['location'].replace("\\", os.sep).replace("/", os.sep)
    location = os.path.join(real_base, relative_location)
    try:
        with open(location, 'r', encoding='utf-8') as f_swcs:
            swc_json = json.load(f_swcs)
            if 'SWCs' in swc_json and isinstance(swc_json['SWCs'], list):
                for swc in swc_json['SWCs']:
                    if 'category' in swc:
                        category_list.append(swc['category'])
    except Exception as e:
        print(f"Could not read {location}: {e}")
taxonomia_raw = sorted(set(category_list))
# Formata a taxonomia como lista numerada para o prompt
taxonomia = '\n'.join([f"{i+1}. {cat}" for i, cat in enumerate(taxonomia_raw)])
#print("Taxonomia extraída e formatada:\n", taxonomia)

##### Classificação multiclasse usando taxonomia

In [ ]:
csv_path = "../../../Datasets/error_classification_task.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset_bug = df.reset_index(drop=True)

In [ ]:
def get_true_category(file_path, bug_locations, real_base):
    """Obtém a categoria real do erro do arquivo SWC"""
    for item in bug_locations:
        if normalize_path(item['file']) in file_path:
            relative_location = item['location'].replace("\\", os.sep).replace("/", os.sep)
            location = os.path.join(real_base, relative_location)
            try:
                with open(location, 'r', encoding='utf-8') as f_swcs:
                    swc_json = json.load(f_swcs)
                    if 'SWCs' in swc_json and isinstance(swc_json['SWCs'], list):
                        # Retorna a primeira categoria encontrada
                        for swc in swc_json['SWCs']:
                            if 'category' in swc:
                                return swc['category']
            except Exception as e:
                print(f"Erro ao ler categoria: {e}")
    return "Categoria não encontrada"

In [ ]:
def classify_multiclass(dataset, taxonomia, MODEL_NAME):
    results = []
    for idx, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Classificando arquivos", ncols=90):
        file_name = row.get('file_name', '')
        file_path = row.get('file_path', '')
        full_path = f'../../../Datasets/{file_path}'
        
        true_category = get_true_category(file_path, bug_locations, real_base)
        prompt = prompts.create_multiclass_prompt(full_path, file_name, taxonomia)
        
        try:
            llm_response = call_openai(MODEL_NAME, prompt)
            
            bug_type = "Não identificado"
            if isinstance(llm_response, str):
                # Tenta encontrar o campo bug_type mesmo que o JSON não seja perfeitamente formatado
                try:
                    response_json = json.loads(llm_response)
                    bug_type = response_json.get("bug_type", "").strip() or "Não identificado"
                except json.JSONDecodeError:
                    # Se não for JSON, tenta extrair via regex
                    import re
                    match = re.search(r'"bug_type"\s*:\s*"?([^"\n}]+)"?', llm_response)
                    if match:
                        bug_type = match.group(1).strip()
                    else:
                        bug_type = llm_response.strip()
            elif isinstance(llm_response, dict):
                bug_type = llm_response.get("bug_type", "Não identificado")
            else:
                bug_type = str(llm_response).strip()
            
            results.append({
                'id': idx,
                'file_path': file_path,
                'file_name': file_name,
                'true_label': true_category,
                'llm_label': bug_type
            })
        
        except Exception as e:
            tqdm.write(f"[Erro no processo {idx}] {e}")
    
    print("\nClassificação concluída.")
    return pd.DataFrame(results)


In [ ]:
multiclass_results_df = classify_multiclass(subset_bug, taxonomia, MODEL_NAME)
multiclass_results_df.head()

In [ ]:
from difflib import get_close_matches

def extract_taxonomy_codes(taxonomia: str):
    """
    Extrai todos os códigos SWC (ex: SWC-101) da taxonomia.
    """
    lst = [x.strip() for x in taxonomia.split('\n') if x.strip()]
    codes = []
    for x in lst:
        m = re.search(r'SWC-\d+', x)
        if m:
            codes.append(m.group(0))
    return codes


def find_full_swc_label(label, taxonomy_lines):
    """
    Retorna o rótulo completo no formato SWC-###-Descrição.
    Corrige casos em que o LLM retornou apenas a descrição textual (ex: 'Typographical Error').
    """
    if not isinstance(label, str):
        return label.strip() if isinstance(label, str) else label

    label_clean = label.strip()

    # 1️⃣ Se já contém SWC-###, retorna normalizado
    swc_match = re.search(r'SWC-\d+', label_clean)
    if swc_match:
        return swc_match.group(0)

    # 2️⃣ Caso contrário, tenta achar uma linha da taxonomia que contenha a descrição
    match = get_close_matches(label_clean.lower(), [t.lower() for t in taxonomy_lines], n=1, cutoff=0.6)
    if match:
        # Recupera a linha original (com SWC completo)
        for line in taxonomy_lines:
            if match[0] in line.lower():
                full_match = re.search(r'SWC-\d+', line)
                if full_match:
                    return full_match.group(0)
                return line.strip()
    
    # 3️⃣ Se nada encontrado, retorna o rótulo original
    return label_clean


def normalize_to_swc(series, taxonomy_text):
    """
    Normaliza uma série de rótulos (strings ou números) para o formato SWC-###.
    """
    taxonomy_lines = [x.strip() for x in taxonomy_text.split('\n') if x.strip()]
    s = pd.Series(series).copy()

    if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s):
        taxonomy_codes = extract_taxonomy_codes(taxonomy_text)
        s = s.astype(int).map(lambda i: taxonomy_codes[i] if 0 <= i < len(taxonomy_codes) else f"UNK-{i}")
    else:
        s = s.astype(str).map(lambda v: find_full_swc_label(v, taxonomy_lines))
    return s


def calculate_detailed_metrics(y_true, y_pred):
    metrics = {}
    for average in ['micro', 'macro', 'weighted']:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=average, zero_division=0
        )
        metrics[average] = {'precision': precision, 'recall': recall, 'f1': f1}

    print("=== Métricas Detalhadas ===")
    for avg_type in metrics:
        print(f"\nMétricas ({avg_type}):")
        print(f"Precisão: {metrics[avg_type]['precision']:.3f}")
        print(f"Recall: {metrics[avg_type]['recall']:.3f}")
        print(f"F1-Score: {metrics[avg_type]['f1']:.3f}")

    return metrics


# ---- Execução Principal ----
taxonomia_codes = extract_taxonomy_codes(taxonomia)

if not multiclass_results_df.empty:
    y_true_codes = normalize_to_swc(multiclass_results_df['true_label'], taxonomia)
    y_pred_codes = normalize_to_swc(multiclass_results_df['llm_label'], taxonomia)

    present_labels = sorted(pd.unique(pd.concat([y_true_codes, y_pred_codes])), key=lambda x: (x.startswith('SWC-'), x))
    labels_plot = [c for c in taxonomia_codes if c in set(present_labels)]
    fallback = [c for c in present_labels if c not in set(labels_plot)]
    labels_plot = labels_plot + fallback

    print("\nResultados da Classificação Multiclasse:")
    _metrics = calculate_detailed_metrics(y_true_codes, y_pred_codes)

    conf_mat = confusion_matrix(y_true_codes, y_pred_codes, labels=labels_plot)

    plt.figure(figsize=(max(8, 0.4*len(labels_plot)), max(6, 0.4*len(labels_plot))))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_plot, yticklabels=labels_plot)
    plt.title('Matriz de Confusão - Classificação Multiclasse')
    plt.ylabel('Verdadeiro (SWC)')
    plt.xlabel('Predito (SWC)')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum caso para análise multiclasse.")


In [ ]:
multiclass_results_df['true_label'].value_counts()

#multiclass_results_df['true_label'].value_counts().count()




#fazer um log dos caras errados em formato de log 


In [ ]:
# --- Normalização usando a função que deu certo ---
y_true_codes = normalize_to_swc(multiclass_results_df['true_label'], taxonomia)
y_pred_codes = normalize_to_swc(multiclass_results_df['llm_label'], taxonomia)

# Substitui as colunas antigas (para facilitar o uso posterior)
multiclass_results_df['true_label_swc'] = y_true_codes
multiclass_results_df['llm_label_swc'] = y_pred_codes

# --- 1️⃣ Contagem de labels preditas ---
print("=== Distribuição das classes preditas (normalizadas) ===\n")
display(multiclass_results_df['llm_label_swc'].value_counts().sort_index())



#### Zero-shot CoT Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_cot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

##### Classificação de erros pela LLM usando taxonomia

Nesta seção, vamos passar a taxonomia de erros para a LLM e pedir para ela classificar os casos de erro, comparando com as labels do dataset.

In [ ]:
# Extração da taxonomia de categorias SWC (baseado em distribuicao.py)
category_list = []
real_base = '../../../Datasets/DAppSCAN-main/DAppSCAN-source/'
for item in bug_locations:
    relative_location = item['location'].replace("\\", os.sep).replace("/", os.sep)
    location = os.path.join(real_base, relative_location)
    try:
        with open(location, 'r', encoding='utf-8') as f_swcs:
            swc_json = json.load(f_swcs)
            if 'SWCs' in swc_json and isinstance(swc_json['SWCs'], list):
                for swc in swc_json['SWCs']:
                    if 'category' in swc:
                        category_list.append(swc['category'])
    except Exception as e:
        print(f"Could not read {location}: {e}")
taxonomia_raw = sorted(set(category_list))
# Formata a taxonomia como lista numerada para o prompt
taxonomia = '\n'.join([f"{i+1}. {cat}" for i, cat in enumerate(taxonomia_raw)])
#print("Taxonomia extraída e formatada:\n", taxonomia)

##### Classificação multiclasse usando taxonomia

In [ ]:
csv_path = "../../../Datasets/error_classification_task.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset_bug = df.reset_index(drop=True)

In [ ]:
def get_true_category(file_path, bug_locations, real_base):
    """Obtém a categoria real do erro do arquivo SWC"""
    for item in bug_locations:
        if normalize_path(item['file']) in file_path:
            relative_location = item['location'].replace("\\", os.sep).replace("/", os.sep)
            location = os.path.join(real_base, relative_location)
            try:
                with open(location, 'r', encoding='utf-8') as f_swcs:
                    swc_json = json.load(f_swcs)
                    if 'SWCs' in swc_json and isinstance(swc_json['SWCs'], list):
                        # Retorna a primeira categoria encontrada
                        for swc in swc_json['SWCs']:
                            if 'category' in swc:
                                return swc['category']
            except Exception as e:
                print(f"Erro ao ler categoria: {e}")
    return "Categoria não encontrada"

In [ ]:
def classify_multiclass(dataset, taxonomia, MODEL_NAME):
    results = []
    for idx, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Classificando arquivos", ncols=90):
        file_name = row.get('file_name', '')
        file_path = row.get('file_path', '')
        full_path = f'../../../Datasets/{file_path}'
        
        true_category = get_true_category(file_path, bug_locations, real_base)
        prompt = prompts.create_multiclass_prompt_cot(full_path, file_name, taxonomia)
        
        try:
            llm_response = call_openai(MODEL_NAME, prompt)
            
            bug_type = "Não identificado"
            if isinstance(llm_response, str):
                # Tenta encontrar o campo bug_type mesmo que o JSON não seja perfeitamente formatado
                try:
                    response_json = json.loads(llm_response)
                    bug_type = response_json.get("bug_type", "").strip() or "Não identificado"
                except json.JSONDecodeError:
                    # Se não for JSON, tenta extrair via regex
                    import re
                    match = re.search(r'"bug_type"\s*:\s*"?([^"\n}]+)"?', llm_response)
                    if match:
                        bug_type = match.group(1).strip()
                    else:
                        bug_type = llm_response.strip()
            elif isinstance(llm_response, dict):
                bug_type = llm_response.get("bug_type", "Não identificado")
            else:
                bug_type = str(llm_response).strip()
            
            results.append({
                'id': idx,
                'file_path': file_path,
                'file_name': file_name,
                'true_label': true_category,
                'llm_label': bug_type
            })
        
        except Exception as e:
            tqdm.write(f"[Erro no processo {idx}] {e}")
    
    print("\nClassificação concluída.")
    return pd.DataFrame(results)


In [ ]:
multiclass_results_df = classify_multiclass(subset_bug, taxonomia, MODEL_NAME)
multiclass_results_df.head()

In [ ]:
from difflib import get_close_matches

def extract_taxonomy_codes(taxonomia: str):
    """
    Extrai todos os códigos SWC (ex: SWC-101) da taxonomia.
    """
    lst = [x.strip() for x in taxonomia.split('\n') if x.strip()]
    codes = []
    for x in lst:
        m = re.search(r'SWC-\d+', x)
        if m:
            codes.append(m.group(0))
    return codes


def find_full_swc_label(label, taxonomy_lines):
    """
    Retorna o rótulo completo no formato SWC-###-Descrição.
    Corrige casos em que o LLM retornou apenas a descrição textual (ex: 'Typographical Error').
    """
    if not isinstance(label, str):
        return label.strip() if isinstance(label, str) else label

    label_clean = label.strip()

    # 1️⃣ Se já contém SWC-###, retorna normalizado
    swc_match = re.search(r'SWC-\d+', label_clean)
    if swc_match:
        return swc_match.group(0)

    # 2️⃣ Caso contrário, tenta achar uma linha da taxonomia que contenha a descrição
    match = get_close_matches(label_clean.lower(), [t.lower() for t in taxonomy_lines], n=1, cutoff=0.6)
    if match:
        # Recupera a linha original (com SWC completo)
        for line in taxonomy_lines:
            if match[0] in line.lower():
                full_match = re.search(r'SWC-\d+', line)
                if full_match:
                    return full_match.group(0)
                return line.strip()
    
    # 3️⃣ Se nada encontrado, retorna o rótulo original
    return label_clean


def normalize_to_swc(series, taxonomy_text):
    """
    Normaliza uma série de rótulos (strings ou números) para o formato SWC-###.
    """
    taxonomy_lines = [x.strip() for x in taxonomy_text.split('\n') if x.strip()]
    s = pd.Series(series).copy()

    if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s):
        taxonomy_codes = extract_taxonomy_codes(taxonomy_text)
        s = s.astype(int).map(lambda i: taxonomy_codes[i] if 0 <= i < len(taxonomy_codes) else f"UNK-{i}")
    else:
        s = s.astype(str).map(lambda v: find_full_swc_label(v, taxonomy_lines))
    return s


def calculate_detailed_metrics(y_true, y_pred):
    metrics = {}
    for average in ['micro', 'macro', 'weighted']:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=average, zero_division=0
        )
        metrics[average] = {'precision': precision, 'recall': recall, 'f1': f1}

    print("=== Métricas Detalhadas ===")
    for avg_type in metrics:
        print(f"\nMétricas ({avg_type}):")
        print(f"Precisão: {metrics[avg_type]['precision']:.3f}")
        print(f"Recall: {metrics[avg_type]['recall']:.3f}")
        print(f"F1-Score: {metrics[avg_type]['f1']:.3f}")

    return metrics


# ---- Execução Principal ----
taxonomia_codes = extract_taxonomy_codes(taxonomia)

if not multiclass_results_df.empty:
    y_true_codes = normalize_to_swc(multiclass_results_df['true_label'], taxonomia)
    y_pred_codes = normalize_to_swc(multiclass_results_df['llm_label'], taxonomia)

    present_labels = sorted(pd.unique(pd.concat([y_true_codes, y_pred_codes])), key=lambda x: (x.startswith('SWC-'), x))
    labels_plot = [c for c in taxonomia_codes if c in set(present_labels)]
    fallback = [c for c in present_labels if c not in set(labels_plot)]
    labels_plot = labels_plot + fallback

    print("\nResultados da Classificação Multiclasse:")
    _metrics = calculate_detailed_metrics(y_true_codes, y_pred_codes)

    conf_mat = confusion_matrix(y_true_codes, y_pred_codes, labels=labels_plot)

    plt.figure(figsize=(max(8, 0.4*len(labels_plot)), max(6, 0.4*len(labels_plot))))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_plot, yticklabels=labels_plot)
    plt.title('Matriz de Confusão - Classificação Multiclasse')
    plt.ylabel('Verdadeiro (SWC)')
    plt.xlabel('Predito (SWC)')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum caso para análise multiclasse.")


In [ ]:
multiclass_results_df['true_label'].value_counts()


In [ ]:
# --- Normalização usando a função que deu certo ---
y_true_codes = normalize_to_swc(multiclass_results_df['true_label'], taxonomia)
y_pred_codes = normalize_to_swc(multiclass_results_df['llm_label'], taxonomia)

# Substitui as colunas antigas (para facilitar o uso posterior)
multiclass_results_df['true_label_swc'] = y_true_codes
multiclass_results_df['llm_label_swc'] = y_pred_codes

# --- 1️⃣ Contagem de labels preditas ---
print("=== Distribuição das classes preditas (normalizadas) ===\n")
display(multiclass_results_df['llm_label_swc'].value_counts().sort_index())



#### Zero-shot ToT Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

##### Classificação de erros pela LLM usando taxonomia

Nesta seção, vamos passar a taxonomia de erros para a LLM e pedir para ela classificar os casos de erro, comparando com as labels do dataset.

In [ ]:
# Extração da taxonomia de categorias SWC (baseado em distribuicao.py)
category_list = []
real_base = '../../../Datasets/DAppSCAN-main/DAppSCAN-source/'
for item in bug_locations:
    relative_location = item['location'].replace("\\", os.sep).replace("/", os.sep)
    location = os.path.join(real_base, relative_location)
    try:
        with open(location, 'r', encoding='utf-8') as f_swcs:
            swc_json = json.load(f_swcs)
            if 'SWCs' in swc_json and isinstance(swc_json['SWCs'], list):
                for swc in swc_json['SWCs']:
                    if 'category' in swc:
                        category_list.append(swc['category'])
    except Exception as e:
        print(f"Could not read {location}: {e}")
taxonomia_raw = sorted(set(category_list))
# Formata a taxonomia como lista numerada para o prompt
taxonomia = '\n'.join([f"{i+1}. {cat}" for i, cat in enumerate(taxonomia_raw)])

##### Classificação multiclasse usando taxonomia

In [ ]:
csv_path = "../../../Datasets/error_classification_task.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset_bug = df.reset_index(drop=True)

In [ ]:
def get_true_category(file_path, bug_locations, real_base):
    """Obtém a categoria real do erro do arquivo SWC"""
    for item in bug_locations:
        if normalize_path(item['file']) in file_path:
            relative_location = item['location'].replace("\\", os.sep).replace("/", os.sep)
            location = os.path.join(real_base, relative_location)
            try:
                with open(location, 'r', encoding='utf-8') as f_swcs:
                    swc_json = json.load(f_swcs)
                    if 'SWCs' in swc_json and isinstance(swc_json['SWCs'], list):
                        # Retorna a primeira categoria encontrada
                        for swc in swc_json['SWCs']:
                            if 'category' in swc:
                                return swc['category']
            except Exception as e:
                print(f"Erro ao ler categoria: {e}")
    return "Categoria não encontrada"

In [ ]:
def classify_multiclass(dataset, taxonomia, MODEL_NAME):
    results = []
    for idx, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Classificando arquivos", ncols=90):
        file_name = row.get('file_name', '')
        file_path = row.get('file_path', '')
        full_path = f'../../../Datasets/{file_path}'
        
        true_category = get_true_category(file_path, bug_locations, real_base)
        prompt = prompts.create_multiclass_prompt_tot(full_path, file_name, taxonomia)
        
        try:
            llm_response = call_openai(MODEL_NAME, prompt)
            
            bug_type = "Não identificado"
            if isinstance(llm_response, str):
                # Tenta encontrar o campo bug_type mesmo que o JSON não seja perfeitamente formatado
                try:
                    response_json = json.loads(llm_response)
                    bug_type = response_json.get("bug_type", "").strip() or "Não identificado"
                except json.JSONDecodeError:
                    # Se não for JSON, tenta extrair via regex
                    import re
                    match = re.search(r'"bug_type"\s*:\s*"?([^"\n}]+)"?', llm_response)
                    if match:
                        bug_type = match.group(1).strip()
                    else:
                        bug_type = llm_response.strip()
            elif isinstance(llm_response, dict):
                bug_type = llm_response.get("bug_type", "Não identificado")
            else:
                bug_type = str(llm_response).strip()
            
            results.append({
                'id': idx,
                'file_path': file_path,
                'file_name': file_name,
                'true_label': true_category,
                'llm_label': bug_type
            })
        
        except Exception as e:
            tqdm.write(f"[Erro no processo {idx}] {e}")
    
    print("\nClassificação concluída.")
    return pd.DataFrame(results)


In [ ]:
multiclass_results_df = classify_multiclass(subset_bug, taxonomia, MODEL_NAME)
multiclass_results_df.head()

In [ ]:
from difflib import get_close_matches

def extract_taxonomy_codes(taxonomia: str):
    """
    Extrai todos os códigos SWC (ex: SWC-101) da taxonomia.
    """
    lst = [x.strip() for x in taxonomia.split('\n') if x.strip()]
    codes = []
    for x in lst:
        m = re.search(r'SWC-\d+', x)
        if m:
            codes.append(m.group(0))
    return codes


def find_full_swc_label(label, taxonomy_lines):
    """
    Retorna o rótulo completo no formato SWC-###-Descrição.
    Corrige casos em que o LLM retornou apenas a descrição textual (ex: 'Typographical Error').
    """
    if not isinstance(label, str):
        return label.strip() if isinstance(label, str) else label

    label_clean = label.strip()

    # 1️⃣ Se já contém SWC-###, retorna normalizado
    swc_match = re.search(r'SWC-\d+', label_clean)
    if swc_match:
        return swc_match.group(0)

    # 2️⃣ Caso contrário, tenta achar uma linha da taxonomia que contenha a descrição
    match = get_close_matches(label_clean.lower(), [t.lower() for t in taxonomy_lines], n=1, cutoff=0.6)
    if match:
        # Recupera a linha original (com SWC completo)
        for line in taxonomy_lines:
            if match[0] in line.lower():
                full_match = re.search(r'SWC-\d+', line)
                if full_match:
                    return full_match.group(0)
                return line.strip()
    
    # 3️⃣ Se nada encontrado, retorna o rótulo original
    return label_clean


def normalize_to_swc(series, taxonomy_text):
    """
    Normaliza uma série de rótulos (strings ou números) para o formato SWC-###.
    """
    taxonomy_lines = [x.strip() for x in taxonomy_text.split('\n') if x.strip()]
    s = pd.Series(series).copy()

    if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s):
        taxonomy_codes = extract_taxonomy_codes(taxonomy_text)
        s = s.astype(int).map(lambda i: taxonomy_codes[i] if 0 <= i < len(taxonomy_codes) else f"UNK-{i}")
    else:
        s = s.astype(str).map(lambda v: find_full_swc_label(v, taxonomy_lines))
    return s


def calculate_detailed_metrics(y_true, y_pred):
    metrics = {}
    for average in ['micro', 'macro', 'weighted']:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=average, zero_division=0
        )
        metrics[average] = {'precision': precision, 'recall': recall, 'f1': f1}

    print("=== Métricas Detalhadas ===")
    for avg_type in metrics:
        print(f"\nMétricas ({avg_type}):")
        print(f"Precisão: {metrics[avg_type]['precision']:.3f}")
        print(f"Recall: {metrics[avg_type]['recall']:.3f}")
        print(f"F1-Score: {metrics[avg_type]['f1']:.3f}")

    return metrics


# ---- Execução Principal ----
taxonomia_codes = extract_taxonomy_codes(taxonomia)

if not multiclass_results_df.empty:
    y_true_codes = normalize_to_swc(multiclass_results_df['true_label'], taxonomia)
    y_pred_codes = normalize_to_swc(multiclass_results_df['llm_label'], taxonomia)

    present_labels = sorted(pd.unique(pd.concat([y_true_codes, y_pred_codes])), key=lambda x: (x.startswith('SWC-'), x))
    labels_plot = [c for c in taxonomia_codes if c in set(present_labels)]
    fallback = [c for c in present_labels if c not in set(labels_plot)]
    labels_plot = labels_plot + fallback

    print("\nResultados da Classificação Multiclasse:")
    _metrics = calculate_detailed_metrics(y_true_codes, y_pred_codes)

    conf_mat = confusion_matrix(y_true_codes, y_pred_codes, labels=labels_plot)

    plt.figure(figsize=(max(8, 0.4*len(labels_plot)), max(6, 0.4*len(labels_plot))))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_plot, yticklabels=labels_plot)
    plt.title('Matriz de Confusão - Classificação Multiclasse')
    plt.ylabel('Verdadeiro (SWC)')
    plt.xlabel('Predito (SWC)')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum caso para análise multiclasse.")


In [ ]:
multiclass_results_df['true_label'].value_counts()


In [ ]:
# --- Normalização usando a função que deu certo ---
y_true_codes = normalize_to_swc(multiclass_results_df['true_label'], taxonomia)
y_pred_codes = normalize_to_swc(multiclass_results_df['llm_label'], taxonomia)

# Substitui as colunas antigas (para facilitar o uso posterior)
multiclass_results_df['true_label_swc'] = y_true_codes
multiclass_results_df['llm_label_swc'] = y_pred_codes

# --- 1️⃣ Contagem de labels preditas ---
print("=== Distribuição das classes preditas (normalizadas) ===\n")
display(multiclass_results_df['llm_label_swc'].value_counts().sort_index())

